In [1]:
import regex as re


PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""


re.findall(PAT,"some text that i'll pre-tokenize")


['some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize']

# Vocabulary Foundations: Function Mappings & Mental Model

## Goal
Build the initial BPE vocabulary: a `dict[int, bytes]` mapping every possible byte value (0–255) to its single-byte representation, plus any special tokens appended at the end.

```
vocab = {
    0:   b'\x00',
    1:   b'\x01',
    ...
    65:  b'A',       # Python displays printable bytes as characters
    ...
    255: b'\xff',
    256: b'<|endoftext|>'   # special token gets the next available ID
}
```

The vocabulary is the **lookup table** used throughout BPE: during training to track which byte sequences exist, and during encoding/decoding to convert between integer IDs and byte sequences.

---

## The Four Functions and When to Use Each

| Function | Direction | Input → Output | Example |
|---|---|---|---|
| `bytes([i])` | int → bytes | `65` → `b'A'` | Build a single-byte vocab entry |
| `chr(i)` | int → str | `65` → `'A'` | Unicode code point to character |
| `ord(c)` | str → int | `'A'` → `65` | Character to Unicode code point |
| `str.encode("utf-8")` | str → bytes | `'hello'` → `b'hello'` | Encode a string as UTF-8 bytes |
| `bytes.decode("utf-8")` | bytes → str | `b'hello'` → `'hello'` | Decode UTF-8 bytes to string |

### Why `bytes([i])` and not `chr(i).encode("utf-8")`?

Both give the same result for values 0–127 (ASCII range), but diverge above 127:

- `bytes([130])` → `b'\x82'`  — always exactly **1 byte** ✓
- `chr(130).encode("utf-8")` → `b'\xc2\x82'` — **2 bytes** ✗

This is because UTF-8 uses multi-byte sequences to represent Unicode code points above 127. Since we want a *byte-level* vocabulary where each entry is exactly one byte, `bytes([i])` is the right tool.

### Why `str.encode("utf-8")` for special tokens?

Special tokens like `<|endoftext|>` are strings, and vocabulary values must be `bytes`. `.encode("utf-8")` is the correct conversion for arbitrary strings since all ASCII characters encode to exactly the same byte values (no surprises), and the result is a proper `bytes` object.

---

## Key Invariant to Check

After building the vocabulary, these should all hold:

```python
len(vocab) == 257                        # 256 bytes + 1 special token
all(isinstance(k, int) for k in vocab)   # keys are ints
all(isinstance(v, bytes) for v in vocab.values())  # values are bytes
len(vocab[65]) == 1                      # single-byte entries are 1 byte
vocab[65] == b'A'                        # byte 65 is ASCII 'A'
vocab[256] == b'<|endoftext|>'           # special token at ID 256
```

In [2]:
vocab = {i: bytes([i]) for i in range(0,256)}
vocab[256] = '<|endoftext|>'

In [3]:
vocab

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'

## Why pre-tokenize: without it, BPE might merge bytes across word boundaries — you'd end up with tokens like "dog." and "dog!" as separate vocabulary entries, even though they differ only in punctuation. Pre-tokenization with the regex ensures merges only happen within each chunk, never across them.

On pretokenization_example.py: PAT and that file solve different problems:

- PAT + re.findall splits text into pre-tokens (the "what to split")
- pretokenization_example.py splits a large file into chunks at <|endoftext|> boundaries so multiple processes can work in parallel (the "how to parallelize")

The two work together: you use find_chunk_boundaries to divide the file across workers, then each worker runs re.findall(PAT, chunk) on its piece. Without the chunking, you'd have to load the entire corpus into memory and process it serially.

In [4]:
text  = "low low low low low lower lower widest widest widest newest newest newest newest newest newest"

In [5]:
# pretokenize: split raw text into chunks before any BPE merging
pre_tokens_xs = re.findall(PAT, text)

In [6]:
from collections import Counter

pre_token_frqs = Counter(pre_tokens_xs)
pre_token_frqs

Counter({' newest': 6, ' low': 4, ' widest': 3, ' lower': 2, 'low': 1})

In [7]:
print('low'.encode('utf8'))
list('low'.encode('utf8')) # the integer values of l, o, w in ASCII/UTF-8

b'low'


[108, 111, 119]

In [13]:

tuple(bytes([i]) for i in 'low'.encode('utf8'))

(b'l', b'o', b'w')

In [9]:
bytes([3])

b'\x03'

In [10]:
bytes(3)

b'\x00\x00\x00'

# `bytes()` Cheat Sheet: The `[]` Matter

## The Core Gotcha

| Expression | Result | Interpretation |
|---|---|---|
| `bytes(3)` | `b'\x00\x00\x00'` | Create **3 zero bytes** — argument is a *length* |
| `bytes([3])` | `b'\x03'` | Create **1 byte with value 3** — argument is a *list of values* |
| `bytes([65])` | `b'A'` | 1 byte with value 65 (ASCII 'A') |
| `bytes([108, 111, 119])` | `b'low'` | 3 bytes with those values |

**Rule:** when you want to create bytes from specific integer values, always wrap in `[]`.

---

## Converting a String to a Tuple of Single-Byte `bytes` Objects

This is the representation BPE needs — each character as its own `bytes` object so adjacent pairs can be inspected and counted:

```
"low"
  ↓  .encode("utf-8")
b'low'                        # bytes object: iterating gives integers [108, 111, 119]
  ↓  tuple(bytes([i]) for i in ...)
(b'l', b'o', b'w')            # tuple of single-byte bytes objects
```

Why a **tuple** and not a list? Tuples are hashable — they can be used as dictionary keys.
You need this when building the `dict[tuple[bytes, ...], int]` frequency table that BPE operates on.

---

## Why Not `chr(i)` or plain strings?

- `chr(108)` → `'l'` — a Python *string*, not bytes. Strings and bytes are different types.
- `"low"[0]` → `'l'` — still a string character, not a bytes object.
- `bytes([108])` → `b'l'` — the actual bytes object you want.

The BPE vocabulary stores `bytes` values throughout, so every representation needs to stay in `bytes`-land.